<div style="padding:1.2rem 1.3rem;border:1px solid #273246;border-radius:18px;background:linear-gradient(135deg,#101722 0%,#172033 55%,#0f3a33 100%);color:#f8fafc;margin-bottom:0.9rem;">
  <div style="display:flex;gap:0.45rem;flex-wrap:wrap;margin-bottom:0.75rem;">
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(255,180,0,0.16);border:1px solid rgba(255,180,0,0.36);font-size:0.76rem;font-weight:600;">Presenter Notebook</span>
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(7,173,248,0.12);border:1px solid rgba(7,173,248,0.34);font-size:0.76rem;font-weight:600;">spec02 - spec19</span>
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(71,180,117,0.16);border:1px solid rgba(71,180,117,0.4);font-size:0.76rem;font-weight:600;">Notebook Demo Lane</span>
  </div>
  <h1 style="margin:0 0 0.35rem 0;color:#ffffff;">v7 Experiment Story Walkthrough</h1>
  <p style="margin:0;max-width:58rem;line-height:1.58;">A single notebook for walking through the spec02 to spec19 experiment story, explaining what C-Kernel-Engine can do today, and handing off into the current notebook demo lane without bouncing between ad hoc files.</p>
</div>

## The Story

C-Kernel-Engine did not arrive at the current v7 surface in one jump. The project moved through a sequence of representation choices: early raw SVG experiments, the shift toward structured atoms, the climb into scene DSLs, the family-specific compiler contracts, and finally the broader bundle and curriculum line.

This notebook tells that story in a way that is easy to narrate on screen. It begins with what the engine can do today, then uses the key experiments to explain why the stack looks the way it does, and finally lands in the notebooks and artifacts that a user can run right now.

The goal is to keep the walkthrough inside one readable surface: one continuous story, a few concrete artifacts when they help, and a clean handoff into the live demo notebooks at the end.


In [1]:
from pathlib import Path
from collections import Counter
import html
import json
import re
import shlex

from IPython.display import HTML, display

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ckernel_engine").exists() and (candidate / "version" / "v7").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not find the C-Kernel-Engine repo root from cwd={Path.cwd()}. "
        "Open this notebook from inside the repo checkout."
    )

REPORTS_ROOT = REPO_ROOT / "version" / "v7" / "reports"
NOTEBOOKS_ROOT = REPO_ROOT / "notebooks" / "v7_training"
MANIFEST_PATH = REPORTS_ROOT / "spec_training_manifest.json"
STORY_HTML_PATH = REPORTS_ROOT / "spec_training_story.html"
QUICKSTART_NOTEBOOK = NOTEBOOKS_ROOT / "02_v7_python_authoring_quickstart.ipynb"
DATASET_NOTEBOOK = NOTEBOOKS_ROOT / "03_v7_dsl_dataset_preparation.ipynb"
ARTIFACT_NOTEBOOK = NOTEBOOKS_ROOT / "04_v7_python_authoring_artifact_walkthrough.ipynb"
THIS_NOTEBOOK = NOTEBOOKS_ROOT / "01_v7_experiment_story_walkthrough.ipynb"
DEMO_RUN_DIR = Path.home() / ".cache" / "ck-engine-v7" / "models" / "train" / "python-ui-notebook-demo"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
specs = [row for row in manifest.get("specs", []) if isinstance(row, dict) and row.get("id")]
legacy_specs = [row for row in manifest.get("legacy_specs", []) if isinstance(row, dict) and row.get("id")]

def spec_sort_key(spec_id):
    match = re.match(r"spec(\d+)([a-z]?)", spec_id or "")
    if not match:
        return (9999, spec_id or "")
    return (int(match.group(1)), match.group(2))


specs_by_id = {row["id"]: row for row in specs}
legacy_by_id = {}
for row in legacy_specs:
    merged = dict(row)
    merged.setdefault("goal", row.get("note") or "Legacy or design-only rung.")
    merged.setdefault("representation", "legacy/design-only")
    merged.setdefault("lesson", row.get("note"))
    legacy_by_id[merged["id"]] = merged

all_specs = {**legacy_by_id, **specs_by_id}

report_files_by_spec = {}
for row in manifest.get("report_files", []):
    if not isinstance(row, dict):
        continue
    spec_id = row.get("spec")
    file_name = row.get("file")
    if spec_id and file_name:
        report_files_by_spec.setdefault(spec_id, []).append(REPORTS_ROOT / file_name)

for spec_id, row in legacy_by_id.items():
    for source in row.get("sources", []):
        path = Path(source)
        report_files_by_spec.setdefault(spec_id, []).append(path)

for spec_id, paths in list(report_files_by_spec.items()):
    deduped = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(path)
    report_files_by_spec[spec_id] = deduped

def path_uri(path):
    if not path:
        return ""
    return Path(path).expanduser().resolve(strict=False).as_uri()


def fmt_pct(value):
    if value is None:
        return "-"
    return f"{value * 100:.1f}%"


def status_style(status):
    mapping = {
        "gold": ("#47b475", "rgba(71,180,117,0.12)"),
        "iterating": ("#8cb4ff", "rgba(7,173,248,0.10)"),
        "regressed": ("#ff9b8c", "rgba(231,76,60,0.10)"),
        "legacy": ("#c2cad6", "rgba(194,202,214,0.10)"),
        "design_only": ("#ffd58a", "rgba(255,180,0,0.12)"),
    }
    return mapping.get(status or "", ("#c2cad6", "rgba(194,202,214,0.10)"))


def spec_row(spec_id):
    row = dict(all_specs.get(spec_id, {"id": spec_id, "name": spec_id}))
    row.setdefault("goal", row.get("note") or "Goal still needs curation in the manifest.")
    row.setdefault("representation", "representation metadata still needs curation")
    row.setdefault("lesson", row.get("note") or "Lesson still needs curation in the manifest.")
    return row


def render_metric_cards(cards):
    card_html = []
    for card in cards:
        color = card.get("color", "#8cb4ff")
        card_html.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;'>
                <div style='font-size:0.74rem;text-transform:uppercase;letter-spacing:0.05em;color:#8b97ab;margin-bottom:0.32rem;'>{html.escape(card['label'])}</div>
                <div style='font-size:1.55rem;font-weight:800;color:{color};margin-bottom:0.24rem;'>{html.escape(str(card['value']))}</div>
                <div style='font-size:0.82rem;line-height:1.45;color:#b6c2d1;'>{html.escape(card['note'])}</div>
            </div>
            """
        )
    return (
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:0.7rem;margin:0.5rem 0 1rem 0;'>"
        + "".join(card_html)
        + "</div>"
    )


def render_notebook_lane():
    entries = [
        ("Story Notebook", THIS_NOTEBOOK, "Narrative spine for spec02-spec19 plus the current notebook lane."),
        ("Quickstart", QUICKSTART_NOTEBOOK, "Materialize a tiny run, train, and prepare the viewer artifacts."),
        ("DSL Dataset Prep", DATASET_NOTEBOOK, "Inspect a seed workspace, stage it into a run, and build the dataset viewer."),
        ("Artifact Walkthrough", ARTIFACT_NOTEBOOK, "Inspect manifests, IR, codegen outputs, reports, and viewer artifacts."),
    ]
    cards = []
    for label, path, note in entries:
        cards.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;'>
                <div style='display:flex;align-items:center;justify-content:space-between;gap:0.5rem;margin-bottom:0.45rem;'>
                    <strong style='color:#f8fafc;font-size:0.96rem;'>{html.escape(label)}</strong>
                    <span style='padding:0.16rem 0.48rem;border-radius:999px;background:rgba(7,173,248,0.10);border:1px solid rgba(7,173,248,0.32);color:#8cb4ff;font-size:0.72rem;font-weight:700;'>Notebook</span>
                </div>
                <div style='font-size:0.8rem;line-height:1.45;color:#b6c2d1;margin-bottom:0.55rem;'>{html.escape(note)}</div>
                <div style='font-size:0.73rem;color:#8b97ab;line-height:1.45;'><code>{html.escape(str(path))}</code></div>
            </div>
            """
        )
    return (
        "<div style='margin:0.35rem 0 1rem 0;'>"
        "<h3 style='margin:0 0 0.35rem 0;color:#0f172a;'>Notebook Demo Lane</h3>"
        "<p style='margin:0 0 0.7rem 0;color:#516072;line-height:1.45;'>Use this story notebook as the narrative front door, then move into the user notebooks for the live product demo.</p>"
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:0.7rem;'>"
        + "".join(cards)
        + "</div></div>"
    )


def render_viewer_surfaces():
    viewer_entries = [
        {
            "label": "IR Hub",
            "kind": "Cross-run",
            "note": "Parent dashboard across runs. Use this when you want to move between experiments and show the durable operator surface.",
            "artifact": "ir_hub.html",
            "cmd": ".venv/bin/python version/v7/tools/open_ir_hub.py --open",
        },
        {
            "label": "IR Visualizer",
            "kind": "Per-run",
            "note": "The single-run deep dive. This is the generated ir_report.html surface for IR, layout, runtime, training, and related artifacts.",
            "artifact": str(DEMO_RUN_DIR / "ir_report.html"),
            "cmd": ".venv/bin/python version/v7/tools/open_ir_visualizer.py --generate --run "
                f"{shlex.quote(str(DEMO_RUN_DIR))} --html-only --strict-run-artifacts --output "
                f"{shlex.quote(str(DEMO_RUN_DIR / 'ir_report.html'))}",
        },
        {
            "label": "Dataset Viewer",
            "kind": "Conditional",
            "note": "The dataset-side surface for staged manifests, samples, and split-aware workspace inspection. It appears when the run carries dataset artifacts.",
            "artifact": str(DEMO_RUN_DIR / "dataset_viewer.html"),
            "cmd": f"xdg-open {shlex.quote(str(DEMO_RUN_DIR / 'dataset_viewer.html'))}",
        },
    ]
    cards = []
    for entry in viewer_entries:
        cards.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;'>
                <div style='display:flex;align-items:center;justify-content:space-between;gap:0.5rem;margin-bottom:0.45rem;flex-wrap:wrap;'>
                    <strong style='color:#f8fafc;font-size:0.96rem;'>{html.escape(entry['label'])}</strong>
                    <span style='padding:0.16rem 0.48rem;border-radius:999px;background:rgba(71,180,117,0.12);border:1px solid rgba(71,180,117,0.35);color:#7bd89b;font-size:0.72rem;font-weight:700;'>{html.escape(entry['kind'])}</span>
                </div>
                <div style='font-size:0.8rem;line-height:1.45;color:#b6c2d1;margin-bottom:0.5rem;'>{html.escape(entry['note'])}</div>
                <div style='font-size:0.73rem;color:#8b97ab;line-height:1.45;margin-bottom:0.55rem;'><strong>Artifact:</strong> <code>{html.escape(entry['artifact'])}</code></div>
                <pre style='margin:0;padding:0.72rem;border-radius:10px;background:#0b111c;color:#dbe3ee;font-size:0.72rem;white-space:pre-wrap;line-height:1.45;'>{html.escape(entry['cmd'])}</pre>
            </div>
            """
        )
    return (
        "<div style='margin:0.35rem 0 1rem 0;'>"
        "<h3 style='margin:0 0 0.35rem 0;color:#0f172a;'>Viewer Surfaces</h3>"
        "<p style='margin:0 0 0.7rem 0;color:#516072;line-height:1.45;'>These are the three surfaces worth naming directly in the walkthrough: the hub across runs, the per-run IR visualizer, and the dataset viewer for runs that carry staged dataset manifests.</p>"
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:0.7rem;'>"
        + "".join(cards)
        + "</div></div>"
    )


def render_phase_cards(phases):
    phase_html = []
    for phase in phases:
        spec_cards = []
        for spec_id in phase["spec_ids"]:
            row = spec_row(spec_id)
            color, bg = status_style(row.get("status"))
            reports = report_files_by_spec.get(spec_id, [])
            spec_cards.append(
                f"""
                <div style='border:1px solid #2b3649;border-radius:12px;background:#101722;padding:0.8rem 0.85rem;'>
                    <div style='display:flex;align-items:center;justify-content:space-between;gap:0.45rem;flex-wrap:wrap;margin-bottom:0.35rem;'>
                        <div>
                            <div style='font-size:0.72rem;color:#8b97ab;text-transform:uppercase;letter-spacing:0.05em;'>{html.escape(spec_id)}</div>
                            <div style='font-size:0.95rem;font-weight:800;color:#f8fafc;'>{html.escape(row.get('name') or spec_id)}</div>
                        </div>
                        <span style='padding:0.14rem 0.46rem;border-radius:999px;background:{bg};border:1px solid {color};color:{color};font-size:0.72rem;font-weight:700;text-transform:uppercase;'>{html.escape(row.get('status') or 'unknown')}</span>
                    </div>
                    <div style='font-size:0.79rem;color:#b6c2d1;line-height:1.45;margin-bottom:0.35rem;'><strong style='color:#e8edf5;'>Goal:</strong> {html.escape(row.get('goal') or 'Goal still needs curation.')}</div>
                    <div style='font-size:0.79rem;color:#b6c2d1;line-height:1.45;margin-bottom:0.35rem;'><strong style='color:#e8edf5;'>Representation:</strong> {html.escape(row.get('representation') or 'representation metadata still needs curation')}</div>
                    <div style='font-size:0.79rem;color:#b6c2d1;line-height:1.45;'><strong style='color:#e8edf5;'>Lesson:</strong> {html.escape(row.get('lesson') or row.get('note') or 'Lesson still needs curation.')}</div>
                    <div style='margin-top:0.45rem;font-size:0.72rem;color:#8b97ab;'>Reports surfaced here: {len(reports)}</div>
                </div>
                """
            )
        phase_html.append(
            f"""
            <section style='border:1px solid #dbe3ee;border-radius:16px;background:#f8fbff;padding:0.95rem 1rem;margin:0 0 0.85rem 0;'>
                <div style='display:flex;align-items:flex-start;justify-content:space-between;gap:0.8rem;flex-wrap:wrap;margin-bottom:0.75rem;'>
                    <div>
                        <h3 style='margin:0 0 0.2rem 0;color:#0f172a;'>{html.escape(phase['title'])}</h3>
                        <p style='margin:0;color:#516072;line-height:1.45;'>{html.escape(phase['summary'])}</p>
                    </div>
                    <div style='max-width:24rem;padding:0.55rem 0.7rem;border-left:4px solid #ffb400;background:rgba(255,180,0,0.08);border-radius:10px;color:#5f4b14;font-size:0.8rem;line-height:1.45;'><strong>What changed here:</strong> {html.escape(phase['key_message'])}</div>
                </div>
                <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:0.7rem;'>
                    {''.join(spec_cards)}
                </div>
            </section>
            """
        )
    return "".join(phase_html)


def render_spotlight_table(spec_ids):
    rows = []
    for spec_id in spec_ids:
        row = spec_row(spec_id)
        best = row.get("best_result") if isinstance(row.get("best_result"), dict) else {}
        color, bg = status_style(row.get("status"))
        rows.append(
            f"""
            <tr>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;'><strong>{html.escape(spec_id)}</strong><div style='margin-top:0.18rem;color:#8b97ab;font-size:0.74rem;'>{html.escape(row.get('name') or spec_id)}</div></td>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;color:#425466;'>{html.escape(row.get('representation') or '-')}</td>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;'><span style='padding:0.14rem 0.46rem;border-radius:999px;background:{bg};border:1px solid {color};color:{color};font-size:0.72rem;font-weight:700;text-transform:uppercase;'>{html.escape(row.get('status') or 'unknown')}</span></td>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;color:#425466;'>{fmt_pct(best.get('exact_rate'))}</td>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;color:#425466;'>{fmt_pct(best.get('renderable_rate'))}</td>
                <td style='padding:0.6rem 0.65rem;border-top:1px solid #e5ebf3;vertical-align:top;color:#425466;line-height:1.45;'>{html.escape(row.get('lesson') or row.get('note') or '-')}</td>
            </tr>
            """
        )
    return (
        "<div style='overflow-x:auto;margin:0.4rem 0 1rem 0;'>"
        "<table style='width:100%;border-collapse:collapse;background:white;border:1px solid #dbe3ee;border-radius:14px;overflow:hidden;'>"
        "<thead><tr style='background:#111722;'>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Spec</th>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Representation</th>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Status</th>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Best Exact</th>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Best Renderable</th>"
        "<th style='text-align:left;padding:0.62rem 0.65rem;color:#ffb400;font-size:0.76rem;text-transform:uppercase;'>Takeaway</th>"
        "</tr></thead><tbody>"
        + "".join(rows)
        + "</tbody></table></div>"
    )


def render_incidents(limit=8):
    entries = []
    for row in manifest.get("incidents", [])[:limit]:
        if not isinstance(row, dict):
            continue
        entries.append(
            f"<li style='margin:0 0 0.4rem 0;'><strong>{html.escape(row.get('date') or 'unknown date')}</strong>"
            f" <span style='color:#8b97ab;'>[{html.escape(row.get('spec') or 'cross-spec')}]</span>"
            f" - {html.escape(row.get('title') or 'untitled incident')}</li>"
        )
    return (
        "<div style='border:1px solid #dbe3ee;border-radius:14px;background:#fff;padding:0.9rem 1rem;margin:0.45rem 0 1rem 0;'>"
        "<h3 style='margin:0 0 0.35rem 0;color:#0f172a;'>Notable incidents from the canonical story ledger</h3>"
        "<p style='margin:0 0 0.65rem 0;color:#516072;line-height:1.45;'>These are useful on camera when you want to explain that the line did not advance monotonically; it learned through regressions and repair cycles.</p>"
        "<ul style='margin:0;padding-left:1.15rem;color:#425466;line-height:1.45;'>"
        + "".join(entries)
        + "</ul></div>"
    )


def render_report_excerpt_cards(items, max_lines=16):
    cards = []
    for item in items:
        path = Path(item['path'])
        suffix = path.suffix.lower()
        excerpt = ""
        if path.exists() and suffix in {'.md', '.json', '.txt'}:
            lines = path.read_text(encoding='utf-8').splitlines()
            excerpt = "\n".join(lines[:max_lines])
            if len(lines) > max_lines:
                excerpt += "\n..."
        elif path.exists() and suffix == '.html':
            excerpt = "HTML report. Use the open command in the demo cells instead of trying to read raw HTML in the notebook."
        else:
            excerpt = "File is missing from the current checkout."
        open_cmd = f"xdg-open {shlex.quote(str(path))}" if path.exists() else "missing file"
        cards.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;'>
                <div style='display:flex;align-items:flex-start;justify-content:space-between;gap:0.5rem;flex-wrap:wrap;margin-bottom:0.45rem;'>
                    <strong style='color:#f8fafc;font-size:0.96rem;'>{html.escape(item['title'])}</strong>
                    <span style='padding:0.14rem 0.46rem;border-radius:999px;background:rgba(7,173,248,0.10);border:1px solid rgba(7,173,248,0.32);color:#8cb4ff;font-size:0.72rem;font-weight:700;'>{html.escape(path.suffix or 'file')}</span>
                </div>
                <div style='font-size:0.8rem;color:#b6c2d1;line-height:1.45;margin-bottom:0.55rem;'>{html.escape(item['note'])}</div>
                <div style='font-size:0.72rem;color:#8b97ab;line-height:1.45;margin-bottom:0.6rem;'><code>{html.escape(str(path))}</code></div>
                <pre style='margin:0 0 0.6rem 0;padding:0.7rem;border-radius:10px;background:#0b111c;color:#dbe3ee;font-size:0.72rem;line-height:1.45;white-space:pre-wrap;'>{html.escape(excerpt)}</pre>
                <div style='font-size:0.72rem;color:#8b97ab;'><strong>Open command:</strong> <code>{html.escape(open_cmd)}</code></div>
            </div>
            """
        )
    return (
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:0.75rem;margin:0.35rem 0 1rem 0;'>"
        + "".join(cards)
        + "</div>"
    )


def render_command_blocks(blocks, *, title, note=None):
    cards = []
    for block in blocks:
        cards.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;'>
                <div style='font-size:0.92rem;font-weight:800;color:#f8fafc;margin-bottom:0.35rem;'>{html.escape(block['label'])}</div>
                <div style='font-size:0.8rem;color:#b6c2d1;line-height:1.45;margin-bottom:0.55rem;'>{html.escape(block['detail'])}</div>
                <pre style='margin:0;padding:0.72rem;border-radius:10px;background:#0b111c;color:#dbe3ee;font-size:0.74rem;white-space:pre-wrap;line-height:1.45;'>{html.escape(block['cmd'])}</pre>
            </div>
            """
        )
    note_html = ""
    if note:
        note_html = f"<p style='margin:0 0 0.7rem 0;color:#516072;line-height:1.45;'>{html.escape(note)}</p>"
    return (
        "<div style='margin:0.35rem 0 1rem 0;'>"
        f"<h3 style='margin:0 0 0.25rem 0;color:#0f172a;'>{html.escape(title)}</h3>"
        + note_html
        + "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:0.75rem;'>"
        + "".join(cards)
        + "</div></div>"
    )


print(f"Repo root: {REPO_ROOT}")
print(f"Loaded {len(specs)} primary specs, {len(legacy_specs)} legacy/design-only specs, and {len(manifest.get('runs', []))} runs.")


Repo root: /home/antshiv/Workspace/C-Kernel-Engine
Loaded 19 primary specs, 3 legacy/design-only specs, and 214 runs.


## Chapter 1: What C-Kernel-Engine Does Today

Today the engine presents a full path from project specification to manifests, IR lowering, generated C runtime output, and durable operator artifacts. The notebooks matter because they are now the front door into that stack rather than a side path or a toy wrapper.

That is the first point to establish on screen: the current user story is already coherent. A user can plan a run, lower it, train it, inspect the generated artifacts, and move through the shared viewers without leaving the same product surface.

It helps to name those viewer surfaces clearly. The <code>IR Hub</code> is the cross-run index, the <code>IR Visualizer</code> is the per-run <code>ir_report.html</code> surface, and the <code>Dataset Viewer</code> is the per-run <code>dataset_viewer.html</code> surface that appears when dataset manifests have been staged into a run.


In [2]:
status_counts = Counter(row.get("status") or "unknown" for row in specs)
cards = [
    {"label": "Primary Specs", "value": len(specs), "note": "Structured experiment families in the canonical manifest.", "color": "#8cb4ff"},
    {"label": "Legacy / Design-only", "value": len(legacy_specs), "note": "Important context specs kept in the story but treated differently.", "color": "#ffd58a"},
    {"label": "Recorded Runs", "value": len(manifest.get('runs', [])), "note": "Run entries aggregated by the training story generator.", "color": "#47b475"},
    {"label": "Report Files", "value": len(manifest.get('report_files', [])), "note": "Canonical writeups and support files surfaced by spec.", "color": "#ff9b8c"},
    {"label": "Gold Specs", "value": status_counts.get('gold', 0), "note": "Perfect-exact or converged families in the current manifest view.", "color": "#47b475"},
    {"label": "Iterating Specs", "value": status_counts.get('iterating', 0), "note": "Families that are useful but still actively improving.", "color": "#8cb4ff"},
]
display(HTML(render_metric_cards(cards)))

engine_surface = """
<div style='border:1px solid #dbe3ee;border-radius:16px;background:#f8fbff;padding:0.95rem 1rem;margin:0.35rem 0 0.9rem 0;'>
  <h3 style='margin:0 0 0.35rem 0;color:#0f172a;'>Current Engine Surface</h3>
  <ul style='margin:0;padding-left:1.15rem;color:#425466;line-height:1.55;'>
    <li><strong>Planning + manifests:</strong> Python and CLI flows both hand off into the same v7 manifest-first pipeline.</li>
    <li><strong>IR lowering:</strong> training runs emit IR forward, backward, layout, and train execution plan artifacts.</li>
    <li><strong>Generated runtime:</strong> v7 lowers to generated C and compiled runtime artifacts rather than a notebook-only execution path.</li>
    <li><strong>Operator artifacts:</strong> IR report, IR hub, dataset viewer, train reports, and run-local JSON/HTML outputs sit under one artifact surface.</li>
    <li><strong>Notebook lane:</strong> the current notebooks are now a guided entrypoint for demos and operators.</li>
  </ul>
</div>
"""
display(HTML(engine_surface))
display(HTML(render_notebook_lane()))
display(HTML(render_viewer_surfaces()))

intro_commands = [
    {
        "label": "Launch This Story Notebook",
        "cmd": f".venv/bin/jupyter lab {THIS_NOTEBOOK}",
        "detail": "Narrative spine for the full walkthrough.",
    },
    {
        "label": "Open The Shared IR Hub",
        "cmd": ".venv/bin/python version/v7/tools/open_ir_hub.py --open",
        "detail": "Best top-level operator view when you want to move across runs.",
    },
    {
        "label": "Regenerate The Demo IR Visualizer",
        "cmd": ".venv/bin/python version/v7/tools/open_ir_visualizer.py --generate --run "
            f"{shlex.quote(str(DEMO_RUN_DIR))} --html-only --strict-run-artifacts --output "
            f"{shlex.quote(str(DEMO_RUN_DIR / 'ir_report.html'))}",
        "detail": "Refresh the per-run ir_report.html surface before opening it on camera.",
    },
    {
        "label": "Open The Canonical Story HTML",
        "cmd": f"xdg-open {STORY_HTML_PATH}",
        "detail": "Useful supporting visual if you want the static generated story page beside the notebook.",
    },
]
display(HTML(render_command_blocks(intro_commands, title="Starting Points", note="These commands anchor the story and open the main artifact surfaces.")))


## Chapter 2: The Experiment Arc In Eras

The useful version of this history is not a flat chronology. It is a sequence of representation changes. The line moves from raw SVG to structured tokens, then to scene DSL text, then to family compilers, and finally toward a more generalized bundle language.

That is why `spec02`, `spec03`, and `spec09` stay in the story even when they are not the cleanest rung families in the current manifest. They explain the pivots, and without those pivots the later gold families look arbitrary instead of earned.


In [3]:
PHASES = [
    {
        "title": "Legacy Lessons And The Structured Pivot",
        "summary": "The line starts with raw SVG ambition, runs into tokenizer and representation failures, and then pivots into structured atoms.",
        "key_message": "The raw-SVG line exposed the tokenizer and representation limits clearly enough that the project pivoted toward more explicit contracts.",
        "spec_ids": ["spec02", "spec03", "spec04"],
    },
    {
        "title": "Structured Scenes And Infographics",
        "summary": "The structured representation begins to work: scenes compose, infographic outputs improve, and the line becomes visibly useful.",
        "key_message": "This is the point where the line stopped being an abstract research track and started producing repeatable visible quality.",
        "spec_ids": ["spec05", "spec06"],
    },
    {
        "title": "Scene DSL Climb",
        "summary": "The representation shifts from structured SVG tokens to scene DSL text, then grows richer, asset-aware, keyed, and finally gold-quality.",
        "key_message": "spec09 is the design bridge in this era, and the compiler plus grammar work mattered even when it was not a normal rung family.",
        "spec_ids": ["spec07", "spec08", "spec09", "spec10", "spec11", "spec12"],
    },
    {
        "title": "Family-Specific Compiler Contracts",
        "summary": "The project specializes into comparison boards, timelines, memory maps, and system diagrams to prove that solved families can carry stronger guarantees.",
        "key_message": "This is the clearest place to describe the engine as a compiler-backed visual DSL system rather than just a text predictor.",
        "spec_ids": ["spec13a", "spec13b", "spec14a", "spec14b", "spec15a", "spec15b"],
    },
    {
        "title": "Generalized Bundle And Unified Curriculum",
        "summary": "The later line tries to generalize solved family compilers into a shared bundle language and broader curriculum story without losing execution discipline.",
        "key_message": "The later specs show the ambition of the line and the remaining work, even though their manifest metadata is not yet as polished as the middle families.",
        "spec_ids": ["spec16", "spec17", "spec18", "spec19"],
    },
]

display(HTML(render_phase_cards(PHASES)))


## Chapter 3: Inflection Points Worth Showing On Camera

Not every rung needs equal time. A few inflection points are enough to make the whole line legible: the failure lesson, the first strong structured line, the design pivot, the gold families, the generalized bundle step, and the later curriculum step.

The table below is the compressed version of those pauses. It lets the story move quickly while still making clear why the representation changed over time.


In [4]:
spotlight_specs = ["spec02", "spec03", "spec06", "spec09", "spec12", "spec14a", "spec15a", "spec16", "spec19"]
display(HTML(render_spotlight_table(spotlight_specs)))
display(HTML(render_incidents(limit=8)))

support_commands = [
    {
        "label": "Open Consolidated Story HTML",
        "cmd": f"xdg-open {STORY_HTML_PATH}",
        "detail": "Use when you want the generated static page beside the notebook during recording.",
    },
    {
        "label": "Open spec02 Report Card",
        "cmd": f"xdg-open {REPORTS_ROOT / 'spec02_svg_training_report_card.html'}",
        "detail": "Useful artifact when explaining what the old raw-SVG line could and could not control.",
    },
    {
        "label": "Open spec04/spec05 Iteration Report",
        "cmd": f"xdg-open {REPORTS_ROOT / 'spec04_spec05_iteration_report_20260311.html'}",
        "detail": "Good bridge artifact for the structured-scene improvement section.",
    },
]
display(HTML(render_command_blocks(support_commands, title="Support Artifacts", note="Use these when one or two concrete artifacts will strengthen the story without opening too many reports at once.")))


Spec,Representation,Status,Best Exact,Best Renderable,Takeaway
spec02SVG Atoms,legacy/design-only,legacy,-,-,"Legacy raw-SVG line with an older probe regime. Keep it in the story, but keep it out of unified structured charts."
spec03Bootstrap Tokenizer Failure,legacy/design-only,legacy,-,-,Representation failure case that taught the tokenizer/contract lesson before the structured scene DSL line stabilized.
spec06Structured Infographics,structured SVG infographics,iterating,91.7%,100.0%,strong – minor gaps remain
spec09Scene DSL v2 Grammar,legacy/design-only,design_only,-,-,"Design/compiler bridge that informed later specs, but not a normal cache-backed training family."
spec12Scene DSL (gold),scene DSL v3 (ctx768),gold,100.0%,100.0%,converged – perfect exact match
spec14aComparison Board Family,family scene DSL (comparison board),gold,100.0%,100.0%,converged – perfect exact match
spec15aMemory Map Family,family scene DSL (memory map),gold,100.0%,100.0%,converged – perfect exact match
spec16Generalized Visual Bundle,generalized visual bundle DSL,iterating,77.8%,86.1%,partial – further iteration needed
spec19spec19,-,iterating,87.5%,93.8%,strong – minor gaps remain


## Chapter 4: Read The Canonical Source Notes Through The Notebook

The source notes stay inside this notebook so the walkthrough does not dissolve into a file hunt. The excerpts give enough context to keep the narrative moving on screen, while the open commands remain available when a deeper source is worth showing.

In practice, one strong source per era is usually enough. That keeps the pacing intact while still grounding the story in the actual experiment record.


In [5]:
excerpt_items = [
    {
        "title": "spec03 Representation Worksheet",
        "path": REPORTS_ROOT / "SPEC03_REPRESENTATION_WORKSHEET.md",
        "note": "Use this when you want to explain the tokenizer and representation failure lesson explicitly.",
    },
    {
        "title": "spec10 DSL Training Playbook",
        "path": REPORTS_ROOT / "SPEC10_DSL_TRAINING_PLAYBOOK_2026-03-17.md",
        "note": "Strong source for the practical training-method section of the story.",
    },
    {
        "title": "spec14a Execution Contract",
        "path": REPORTS_ROOT / "SPEC14A_COMPARISON_BOARD_EXECUTION_CONTRACT_2026-03-26.md",
        "note": "Good family-specific proof point for compiler-backed contracts.",
    },
    {
        "title": "spec15a Memory Map Quickstart",
        "path": REPORTS_ROOT / "SPEC15A_MEMORY_MAP_QUICKSTART_2026-03-28.md",
        "note": "Useful when you want a concrete example of a solved family becoming operationally legible.",
    },
    {
        "title": "spec16 Training Decision",
        "path": REPORTS_ROOT / "spec16_training_decision.md",
        "note": "Helps explain the generalized visual bundle line and its tradeoffs.",
    },
    {
        "title": "spec19 Curriculum Blueprint",
        "path": REPORTS_ROOT / "spec19_curriculum_blueprint.json",
        "note": "Good source for explaining the later unified-curriculum framing even though the display metadata is still sparse.",
    },
]

display(HTML(render_report_excerpt_cards(excerpt_items, max_lines=18)))


## Chapter 5: Live Demo Handoff

By the end of the story, the audience should be ready to see the current workflow rather than hear one more summary. This final section turns the historical arc into a present-day demo path.

The clean sequence is simple: start in the story notebook, move into the quickstart, open the per-run IR visualizer, show dataset preparation when you want to explain the data side, open the dataset viewer when that run carries staged manifests, inspect the artifact walkthrough once a run exists, and then return to the IR hub as the durable operator surface.


In [6]:
notebook_commands = [
    {
        "label": "Launch Quickstart Notebook",
        "cmd": f".venv/bin/jupyter lab {QUICKSTART_NOTEBOOK}",
        "detail": "Use this to show Python authoring, materialize/train, and the artifact dashboard.",
    },
    {
        "label": "Launch DSL Dataset Prep Notebook",
        "cmd": f".venv/bin/jupyter lab {DATASET_NOTEBOOK}",
        "detail": "Use this to explain staged dataset workspaces, manifests, dataset_viewer.html, and the training handoff.",
    },
    {
        "label": "Launch Artifact Walkthrough Notebook",
        "cmd": f".venv/bin/jupyter lab {ARTIFACT_NOTEBOOK}",
        "detail": "Use this for the deeper artifact-tour part of the video after the quickstart has produced a run dir.",
    },
    {
        "label": "Open Shared IR Hub",
        "cmd": ".venv/bin/python version/v7/tools/open_ir_hub.py --open",
        "detail": "Best way to jump across runs and surface the durable operator UI after the notebook demo cells finish.",
    },
]
display(HTML(render_command_blocks(notebook_commands, title="Notebook Demo Commands", note="These are the exact commands for the live portion of the video.")))

demo_artifact_commands = [
    {
        "label": "Generate Quickstart IR Visualizer",
        "cmd": ".venv/bin/python version/v7/tools/open_ir_visualizer.py --generate --run "
            f"{shlex.quote(str(DEMO_RUN_DIR))} --html-only --strict-run-artifacts --output "
            f"{shlex.quote(str(DEMO_RUN_DIR / 'ir_report.html'))}",
        "detail": "Use this if you want to refresh the per-run viewer before opening it during the demo.",
    },
    {
        "label": "Open Quickstart IR Visualizer",
        "cmd": f"xdg-open {DEMO_RUN_DIR / 'ir_report.html'}",
        "detail": "This is the per-run IR visualizer surface: the main deep dive once the quickstart run exists.",
    },
    {
        "label": "Open Dataset Viewer If Present",
        "cmd": f"xdg-open {DEMO_RUN_DIR / 'dataset_viewer.html'}",
        "detail": "This will only exist for runs with staged dataset manifests. Use the dataset-prep notebook when you want this output intentionally.",
    },
    {
        "label": "Open Shared IR Hub",
        "cmd": ".venv/bin/python version/v7/tools/open_ir_hub.py --open",
        "detail": "Return here when you want to zoom back out from one run to the broader run landscape.",
    },
]
display(HTML(render_command_blocks(demo_artifact_commands, title="Artifact Commands", note="The dashboard inside JupyterLab is a strong status surface, but on camera explicit open commands are more reliable than local-file clicks.")))

closeout = """
<div style='padding:0.9rem 1rem;border-left:4px solid #47b475;background:rgba(71,180,117,0.08);border-radius:12px;color:#29463a;'>
  <strong>Where the story lands:</strong> the journey moves from representation discipline to compiler-backed families to an artifact-rich operator flow, with notebooks now acting as the front door into the current product surface.
</div>
"""
display(HTML(closeout))
